# KNEEP on thermodynamically consistent lattice ABP

This demo simulates a hard-exclusion lattice active Brownian-particle (TC-LABP) model, trains the repository's **same ShellForce KNEEP** estimator without EP labels, and evaluates it on an independently simulated test set. The implementation follows Kim, Kwon, and Baek, [*Thermodynamically consistent lattice Monte Carlo method for active particles* (arXiv:2503.16958v1)](https://arxiv.org/abs/2503.16958v1), with the accompanying CUDA code pinned at [Ki-Won/Active-particle-lattice-sim@5a3aab5](https://github.com/Ki-Won/Active-particle-lattice-sim/tree/5a3aab57a41870aa7e7f7b490bcbb234eac47e0d). This KNEEP implementation deliberately has one Torch simulation path and hard exclusion only: there is no WCA potential, C0/Cv selector, backend selector, or model-type selector.

For every accepted hop, `exact_ep` records the reservoir (medium) entropy increment, i.e. the log forward/reverse hop-probability ratio. Symmetric angular diffusion contributes zero to this medium term. Over a saved interval, all microscopic hop increments are summed. This is **not** pathwise total EP: system entropy is omitted, though its mean rate vanishes in a stationary state. KNEEP instead estimates the irreversibility visible in two saved observation frames, so exact microscopic medium EP is held out for diagnostics and is not a supervised pairwise label.

`INCLUDE_ANGLE=True` encodes $[\rho,\rho\cos\theta,\rho\sin\theta]$ and contracts ShellForce with all three increments, $F_\rho\Delta\rho+F_x\Delta(\rho\cos\theta)+F_y\Delta(\rho\sin\theta)$; `False` uses only $F_\rho\Delta\rho$. This branch-cut-free polarization contraction lets angular changes contribute to the joint-state KNEEP score. Following the paper, orientation is even under time reversal. The channel decomposition is coordinate dependent and is not, by itself, an exact separation of system and medium EP; an exact boundary term $\Delta s_\mathrm{sys}=\log p_\mathrm{ss}(X_t)-\log p_\mathrm{ss}(X_{t+1})$ would additionally require the stationary state distribution. True local medium EP is placed in the **destination-site gauge** used by the simulator; a learned convolutional map has its own gauge. Sitewise comparisons are therefore descriptive, while spatial sums are the gauge-robust quantities.

In [3]:
# ------------------------- top-level user settings ------------------------
INCLUDE_ANGLE = True

KNEEP_PATH = "/home/ldh041203/projects/KNEEP"

# None: simulate and save; path string: load train/validation/test from that file.
LOAD_TRAJECTORY_PATH = None
# Example:
# LOAD_TRAJECTORY_PATH = r'C:\path\to\tc_labp_trajectories.pt'

## 0. Environment and repository discovery

The repository root is found from the current working directory (or a `KNEEP` child), so the notebook can be launched from the repo, `demos/`, or the shared workspace. The device is selected automatically.

In [4]:
from __future__ import annotations

from dataclasses import asdict
from datetime import datetime
from pathlib import Path
import json
import math
import random
import sys
import warnings

import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, display


def find_kneep_root(start: Path | None = None) -> Path:
    start = (Path.cwd() if start is None else Path(start)).resolve()
    candidates = []
    for parent in (start, *start.parents):
        candidates.extend((parent, parent / 'KNEEP'))
    for candidate in candidates:
        if (candidate / 'models' / 'tc_labp.py').is_file() and (candidate / 'shell_force.py').is_file():
            return candidate
    raise RuntimeError('Could not find the KNEEP repository from the current working directory.')


ROOT = find_kneep_root(KNEEP_PATH)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.tc_labp import (
    TCLABPConfig, TCLABPTrajectory, encode_observations, simulate_trajectories,
)
from utils.training import (
    TrainingConfig,
    predict_epr_component_maps,
    predict_epr_increments,
    train_model,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_STAMP = datetime.now().strftime('%Y-%m-%d-%H%M%S')
RESULT_DIR = ROOT / 'results' / 'tc_labp_demo' / RUN_STAMP
RESULT_DIR.mkdir(parents=True, exist_ok=False)
plt.rcParams.update({'figure.dpi': 110, 'animation.html': 'jshtml'})

print('repo:', ROOT)
print('device:', DEVICE)
print('results:', RESULT_DIR)

repo: /home/ldh041203/projects/KNEEP
device: cuda
results: /home/ldh041203/projects/KNEEP/results/tc_labp_demo/2026-09-03-202214


## 1. Simulation and training configuration

`train_steps`, `validation_steps`, and `test_steps` are numbers of recorded transitions; each trajectory therefore contains `steps + 1` frames. `sampling_steps` is the number of microscopic MC steps accumulated into one recorded transition, so rates below use `dt_recorded = sampling_steps * config.dt`. Train, validation, and test replicas are generated by separate simulator calls with separate seeds.

In [ ]:
physics_config = TCLABPConfig(
    lattice_size=30, density=0.30, speed=50,
    rotational_diffusion=1.5, translational_diffusion=1.0,
    dt=1e-4, lattice_spacing=1.0,
)
data_config = dict(
    train_trajectories=64, validation_trajectories=16, test_trajectories=1,
    train_steps=10_000, validation_steps=10_000, test_steps=20_000,
    burn_steps=100_000, sampling_steps=5,
)
training_config = TrainingConfig(
    alpha=-0.5, iterations=10000, train_batch_size=4096,
    validation_batch_size=4096, prediction_batch_size=256,
    learning_rate=3.0e-4, weight_decay=1.0e-6,
    gradient_clip=1.0, validate_every=100,
)
model_config = dict(hidden_channels=8, hidden_layers=4, max_distance=5)
SEEDS = {'train': 17, 'validation': 29, 'test': 43, 'model': 71}
DT_RECORDED = data_config['sampling_steps'] * physics_config.dt

print(f'include_angle={INCLUDE_ANGLE}, load_trajectory={LOAD_TRAJECTORY_PATH}')
print(json.dumps({'physics': asdict(physics_config), 'data': data_config,
                  'training': asdict(training_config), 'model': model_config}, indent=2))
print(f'n_particles={physics_config.n_particles}, Pe={physics_config.Pe:.6g}, '
      f'max_hop_probability={physics_config.max_hop_probability:.6g}, dt_recorded={DT_RECORDED:g}')

include_angle=True, load_trajectory=None
{
  "physics": {
    "lattice_size": 30,
    "density": 0.3,
    "speed": 50.0,
    "rotational_diffusion": 1.5,
    "translational_diffusion": 1.0,
    "dt": 0.0001,
    "lattice_spacing": 1.0
  },
  "data": {
    "train_trajectories": 64,
    "validation_trajectories": 16,
    "test_trajectories": 1,
    "train_steps": 50000,
    "validation_steps": 50000,
    "test_steps": 100000,
    "burn_steps": 100000,
    "sampling_steps": 5
  },
  "training": {
    "alpha": -0.5,
    "iterations": 10000,
    "train_batch_size": 4096,
    "validation_batch_size": 4096,
    "prediction_batch_size": 256,
    "learning_rate": 0.0003,
    "weight_decay": 1e-06,
    "gradient_clip": 1.0,
    "validate_every": 100,
    "train_fraction": 0.8
  },
  "model": {
    "hidden_channels": 8,
    "hidden_layers": 4,
    "max_distance": 5
  }
}
n_particles=270, Pe=50, max_hop_probability=0.00707107, dt_recorded=0.0005


: 

## 2. Independent train, validation, and test trajectories

Exact EP is simulated and retained for audit/evaluation only. It never enters `train_model` or validation checkpoint selection. For `S` recorded steps there are `T=S+1` frames. Arrays are ensemble-major: fields are `[M,T,C,L,L]`, totals are `[M,S]`, and local maps are `[M,S,L,L]`.

Set `LOAD_TRAJECTORY_PATH` in the first code cell to a previously saved `tc_labp_trajectories.pt` file to load its train/validation/test data. If it is `None`, this cell simulates all three splits and saves them together as `tc_labp_trajectories.pt` in the current run's results directory. The saved file includes the physics/data configuration and seeds; these are restored automatically when loading.

In [6]:
TRAJECTORY_FIELDS = (
    'sites', 'angles', 'occupancy', 'exact_ep', 'exact_ep_maps', 'accepted_hops', 'times',
)
TRAJECTORY_SAVE_PATH = RESULT_DIR / 'tc_labp_trajectories.pt'
TRAJECTORY_LOADED_FROM = None
TRAJECTORY_SAVED_TO = None


def trajectory_dict(result):
    return {field: getattr(result, field) for field in TRAJECTORY_FIELDS}


def trajectory_from_dict(values):
    return TCLABPTrajectory(**{field: values[field] for field in TRAJECTORY_FIELDS})


def simulate_split(name: str, n_trajectories: int, n_steps: int, seed: int):
    print(f'simulating {name}: M={n_trajectories}, S={n_steps}, T={n_steps + 1}, seed={seed}')
    result = simulate_trajectories(
        physics_config,
        n_trajectories=n_trajectories,
        n_steps=n_steps,
        burn_steps=data_config['burn_steps'],
        sampling_steps=data_config['sampling_steps'],
        seed=seed,
        simulation_device=DEVICE,
        storage_dtype=torch.float32,
        progress=True,
    )
    return result


if LOAD_TRAJECTORY_PATH is not None:
    load_path = Path(LOAD_TRAJECTORY_PATH).expanduser()
    if not load_path.is_absolute():
        load_path = ROOT / load_path
    payload = torch.load(load_path, map_location='cpu', weights_only=True)
    if payload.get('schema_version') != 1:
        raise ValueError(f'unsupported trajectory file: {load_path}')
    physics_config = TCLABPConfig(**payload['physics'])
    data_config = payload['data']
    SEEDS = payload['seeds']
    DT_RECORDED = data_config['sampling_steps'] * physics_config.dt
    train_result = trajectory_from_dict(payload['train'])
    validation_result = trajectory_from_dict(payload['validation'])
    test_result = trajectory_from_dict(payload['test'])
    TRAJECTORY_LOADED_FROM = str(load_path.resolve())
    print(f'loaded train/validation/test: {load_path} ({load_path.stat().st_size / 1024**2:.1f} MiB)')
else:
    train_result = simulate_split(
        'train', data_config['train_trajectories'], data_config['train_steps'], SEEDS['train']
    )
    validation_result = simulate_split(
        'validation', data_config['validation_trajectories'],
        data_config['validation_steps'], SEEDS['validation']
    )
    test_result = simulate_split(
        'test', data_config['test_trajectories'], data_config['test_steps'], SEEDS['test']
    )
    torch.save(
        {
            'schema_version': 1,
            'physics': asdict(physics_config),
            'data': data_config,
            'seeds': SEEDS,
            'train': trajectory_dict(train_result),
            'validation': trajectory_dict(validation_result),
            'test': trajectory_dict(test_result),
        },
        TRAJECTORY_SAVE_PATH,
    )
    TRAJECTORY_SAVED_TO = str(TRAJECTORY_SAVE_PATH.resolve())
    print(
        f'saved train/validation/test: {TRAJECTORY_SAVE_PATH} '
        f'({TRAJECTORY_SAVE_PATH.stat().st_size / 1024**2:.1f} MiB)'
    )


def audit_simulation(result, expected_m: int, expected_steps: int, label: str) -> None:
    expected_frames = expected_steps + 1
    L = physics_config.lattice_size
    N = physics_config.n_particles
    assert result.sites.shape == (expected_m, expected_frames, N, 2)
    assert result.angles.shape == (expected_m, expected_frames, N)
    assert result.occupancy.shape == (expected_m, expected_frames, L, L)
    assert result.exact_ep.shape == (expected_m, expected_steps)
    assert result.exact_ep_maps.shape == (expected_m, expected_steps, L, L)
    assert result.accepted_hops.shape == (expected_m, expected_steps)
    assert result.times.shape == (expected_frames,)
    assert bool(torch.isfinite(result.exact_ep).all())
    assert bool(torch.isfinite(result.exact_ep_maps).all())
    assert bool(((result.occupancy == 0) | (result.occupancy == 1)).all())
    assert bool((result.occupancy.sum(dim=(-2, -1)) == N).all())
    torch.testing.assert_close(
        result.exact_ep_maps.sum(dim=(-2, -1)), result.exact_ep, rtol=2e-5, atol=2e-5
    )
    print(label, 'occupancy/exact EP:', tuple(result.occupancy.shape), tuple(result.exact_ep.shape))


audit_simulation(train_result, data_config['train_trajectories'], data_config['train_steps'], 'train')
audit_simulation(validation_result, data_config['validation_trajectories'],
                 data_config['validation_steps'], 'validation')
audit_simulation(test_result, data_config['test_trajectories'], data_config['test_steps'], 'test')

train_video = encode_observations(train_result, include_angle=INCLUDE_ANGLE)
validation_video = encode_observations(validation_result, include_angle=INCLUDE_ANGLE)
test_video = encode_observations(test_result, include_angle=INCLUDE_ANGLE)
N_COMPONENTS = 3 if INCLUDE_ANGLE else 1
CHANNEL_NAMES = ['rho', 'rho cos(theta)', 'rho sin(theta)'] if INCLUDE_ANGLE else ['rho']
CHANNEL_KEYS = ['rho', 'rho_cos_theta', 'rho_sin_theta'] if INCLUDE_ANGLE else ['rho']
EP_COMPONENT_INDICES = tuple(range(N_COMPONENTS))
assert train_video.shape[2] == validation_video.shape[2] == test_video.shape[2] == N_COMPONENTS
assert train_video.dtype == validation_video.dtype == test_video.dtype == torch.float32
print('observation/contraction channels:', CHANNEL_NAMES)
print('train/validation/test videos:', train_video.shape, validation_video.shape, test_video.shape)

simulating train: M=64, S=50000, T=50001, seed=17


simulating validation: M=16, S=50000, T=50001, seed=29


simulating test: M=1, S=100000, T=100001, seed=43


saved train/validation/test: /home/ldh041203/projects/KNEEP/results/tc_labp_demo/2026-09-03-202214/tc_labp_trajectories.pt (38757.8 MiB)
train occupancy/exact EP: (64, 50001, 30, 30) (64, 50000)
validation occupancy/exact EP: (16, 50001, 30, 30) (16, 50000)
test occupancy/exact EP: (1, 100001, 30, 30) (1, 100000)


: 

: 

## 3. Test trajectory animation

Before training, the first independent test replica is shown to verify the simulated particle dynamics. With angle enabled, arrows show particle orientations. Frames are uniformly subsampled to at most 120 and rendered with `FuncAnimation.to_jshtml()`, so no ffmpeg installation is needed.

In [ ]:
def animation_frames(length: int, maximum: int = 120) -> np.ndarray:
    count = min(length, maximum)
    return np.unique(np.rint(np.linspace(0, length - 1, count)).astype(int))


M, T, _, H, W = test_video.shape
P = T - 1
times = test_result.times.detach().cpu().numpy()
time_mid = 0.5 * (times[:-1] + times[1:])
occupancy0 = test_result.occupancy[0].detach().cpu().numpy()
sites0 = test_result.sites[0].detach().cpu().numpy()
angles0 = test_result.angles[0].detach().cpu().numpy()
state_frame_ids = animation_frames(T)

fig, ax = plt.subplots(figsize=(5.6, 5.2), constrained_layout=True)
image_artist = ax.imshow(occupancy0[0].T, origin='lower', cmap='Greys', vmin=0, vmax=1,
                         interpolation='nearest', extent=(-0.5, W - 0.5, -0.5, H - 0.5))
quiver_artist = None
if INCLUDE_ANGLE:
    quiver_artist = ax.quiver(
        sites0[0, :, 0], sites0[0, :, 1], np.cos(angles0[0]), np.sin(angles0[0]),
        color='tab:orange', pivot='mid', angles='xy', scale_units='xy', scale=1.6, width=0.008,
    )
ax.set(xlim=(-0.5, W - 0.5), ylim=(-0.5, H - 0.5), xlabel='x site', ylabel='y site')


def update_trajectory(frame_index: int):
    image_artist.set_data(occupancy0[frame_index].T)
    artists = [image_artist]
    if quiver_artist is not None:
        quiver_artist.set_offsets(sites0[frame_index])
        quiver_artist.set_UVC(np.cos(angles0[frame_index]), np.sin(angles0[frame_index]))
        artists.append(quiver_artist)
    ax.set_title(f'test trajectory 0 | frame {frame_index}/{T - 1} | t={times[frame_index]:.4g}')
    return artists


trajectory_animation = animation.FuncAnimation(
    fig, update_trajectory, frames=state_frame_ids, interval=80, blit=False
)
display(HTML(trajectory_animation.to_jshtml()))
plt.close(fig)

## 4. True local medium-EP animation

Still before training, the simulator's exact medium-EP map for the same test trajectory is shown as a second simulation check. Its symmetric color range is the 99th percentile of the nonzero absolute true-map values; outliers remain in the data and are clipped only in the display. The map uses the simulator's destination-site gauge.

In [ ]:
true_local = test_result.exact_ep_maps.detach().cpu().numpy()
true_abs_nonzero = np.abs(true_local[0]).ravel()
true_abs_nonzero = true_abs_nonzero[true_abs_nonzero > 0]
TRUE_EP_VMAX = (max(float(np.quantile(true_abs_nonzero, 0.99)), 1e-8)
                if true_abs_nonzero.size else 1.0)
pair_frame_ids = animation_frames(P)


def make_ep_animation(maps: np.ndarray, title: str, vmax: float, colorbar_label: str):
    fig, ax = plt.subplots(figsize=(5.8, 5.2), constrained_layout=True)
    artist = ax.imshow(maps[0].T, origin='lower', cmap='RdBu_r',
                       vmin=-vmax, vmax=vmax, interpolation='nearest')
    colorbar = fig.colorbar(artist, ax=ax, shrink=0.82)
    colorbar.set_label(colorbar_label)
    ax.set(xlabel='x site', ylabel='y site')

    def update(frame_index: int):
        artist.set_data(maps[frame_index].T)
        ax.set_title(f'{title} | pair {frame_index}/{P - 1} | t_mid={time_mid[frame_index]:.4g}\n'
                     f'map sum={maps[frame_index].sum():+.4g}')
        return [artist]

    movie = animation.FuncAnimation(
        fig, update, frames=pair_frame_ids, interval=80, blit=False
    )
    html = HTML(movie.to_jshtml())
    plt.close(fig)
    return movie, html


true_ep_animation, true_ep_html = make_ep_animation(
    true_local[0], 'true medium EP (destination gauge)', TRUE_EP_VMAX,
    'true medium-EP increment per destination site',
)
display(true_ep_html)
print(f'true EP animation scale: [-{TRUE_EP_VMAX:.6g}, +{TRUE_EP_VMAX:.6g}]')

## 5. Unsupervised ShellForce KNEEP training

All observation channels enter the same ShellForce network and the final antisymmetric contraction. With angle enabled, `EP_COMPONENT_INDICES=(0, 1, 2)` gives $F_\rho\Delta\rho+F_x\Delta(\rho\cos\theta)+F_y\Delta(\rho\sin\theta)$; without angle it is `(0,)`. The best checkpoint is chosen solely by the independent validation trajectory objective.

In [ ]:
random.seed(SEEDS['model'])
np.random.seed(SEEDS['model'])
torch.manual_seed(SEEDS['model'])
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEEDS['model'])

training_result = train_model(
    train_video=train_video,
    validation_video=validation_video,
    config=training_config,
    model_seed=SEEDS['model'],
    device=DEVICE,
    n_components=N_COMPONENTS,
    hidden_channels=model_config['hidden_channels'],
    hidden_layers=model_config['hidden_layers'],
    max_distance=model_config['max_distance'],
    ep_component_indices=EP_COMPONENT_INDICES,
    progress=True,
)
print(f'best iteration={training_result.best_iteration}, '
      f'validation loss={training_result.best_validation_loss:+.6e}')
if training_result.best_validation_loss >= -1e-6:
    warnings.warn('Validation objective did not improve over the zero-score baseline; '
                  'downstream plots should be treated as a failed-learning diagnostic.')

## 6. Ordered prediction, exact alignment, and scale audits

`predict_epr_component_maps` returns raw maps in ensemble-major, time-minor order. For raw shape `[pair, shell, component, H, W]`, the conversions used below are

```python
pred_component_local = raw.sum(1) / (H * W)
pred_component_total = raw.mean((-2, -1)).sum(1)
pred_local = pred_component_local.sum(1)
pred_total = pred_component_total.sum(1)
```

Thus every component map and their learned total satisfy the corresponding map-sum identities. Exact totals/maps are flattened in the same ensemble-major, time-minor order and asserted before any comparison. Component contributions are useful diagnostics but depend on the chosen polarization coordinates.

In [ ]:
prediction_batch_size = training_config.prediction_batch_size
pred_direct = predict_epr_increments(training_result, test_video, prediction_batch_size, DEVICE)
raw_component_maps = predict_epr_component_maps(
    training_result, test_video, prediction_batch_size, DEVICE
)

M, T, _, H, W = test_video.shape
P = T - 1
assert P == data_config['test_steps']
K = model_config['max_distance'] + 1
Q = len(EP_COMPONENT_INDICES)
assert raw_component_maps.shape == (M * P, K, Q, H, W)
assert pred_direct.shape == (M * P,)
assert np.isfinite(raw_component_maps).all() and np.isfinite(pred_direct).all()

raw_branch_maps = raw_component_maps.sum(axis=2)
component_branch_total_flat = raw_component_maps.mean(axis=(-2, -1))
component_local_flat = raw_component_maps.sum(axis=1) / (H * W)
component_total_flat = component_branch_total_flat.sum(axis=1)
pred_local_flat = component_local_flat.sum(axis=1)
pred_total_flat = component_total_flat.sum(axis=1)
pred_local = pred_local_flat.reshape(M, P, H, W)
pred_total = pred_total_flat.reshape(M, P)
branch_total_flat = component_branch_total_flat.sum(axis=2)
component_local = component_local_flat.reshape(M, P, Q, H, W)
component_total = component_total_flat.reshape(M, P, Q)

true_total = test_result.exact_ep.detach().cpu().numpy()
true_local = test_result.exact_ep_maps.detach().cpu().numpy()
accepted_hops = test_result.accepted_hops.detach().cpu().numpy()
times = test_result.times.detach().cpu().numpy()
true_total_flat = true_total.reshape(-1)  # ensemble-major, then time-minor
true_local_flat = true_local.reshape(-1, H, W)

assert true_total.shape == pred_total.shape == (M, P)
assert component_total.shape == (M, P, Q)
assert component_local.shape == (M, P, Q, H, W)
assert true_local.shape == pred_local.shape == (M, P, H, W)
np.testing.assert_array_equal(true_total_flat[:P], true_total[0])
np.testing.assert_array_equal(true_local_flat[:P], true_local[0])
np.testing.assert_allclose(true_local_flat.sum(axis=(-2, -1)), true_total_flat, rtol=2e-5, atol=2e-5)
np.testing.assert_allclose(component_local_flat.sum(axis=(-2, -1)), component_total_flat,
                           rtol=2e-5, atol=2e-5)
np.testing.assert_allclose(component_total_flat.sum(axis=1), pred_total_flat, rtol=2e-5, atol=2e-5)
np.testing.assert_allclose(pred_local_flat.sum(axis=(-2, -1)), pred_total_flat, rtol=2e-5, atol=2e-5)
np.testing.assert_allclose(component_branch_total_flat.sum(axis=2), branch_total_flat,
                           rtol=2e-5, atol=2e-5)
np.testing.assert_allclose(branch_total_flat.sum(axis=1), pred_total_flat, rtol=2e-5, atol=2e-5)
np.testing.assert_allclose(pred_direct, pred_total_flat, rtol=2e-5, atol=2e-5)


def alpha_neep_objective(scores: np.ndarray, alpha: float) -> float:
    values = np.asarray(scores, dtype=np.float64)
    if alpha == 0.0:
        return float(np.mean(-values + np.exp(-values) - 1.0))
    if alpha == -1.0:
        raise ValueError('alpha=-1 is singular')
    terms = -np.expm1(alpha * values) / alpha
    terms += np.expm1(-(1.0 + alpha) * values) / (1.0 + alpha)
    return float(np.mean(terms))


def comparison_metrics(target: np.ndarray, prediction: np.ndarray, interval: float) -> dict:
    target = np.asarray(target, dtype=np.float64).ravel()
    prediction = np.asarray(prediction, dtype=np.float64).ravel()
    if target.shape != prediction.shape or target.size == 0:
        raise ValueError(f'invalid metric arrays: {target.shape} vs {prediction.shape}')
    if not (np.isfinite(target).all() and np.isfinite(prediction).all()):
        raise ValueError('metrics require finite arrays')
    residual = prediction - target
    centered_ss = float(np.sum((target - target.mean()) ** 2))
    pearson = (float(np.corrcoef(target, prediction)[0, 1])
               if target.size > 1 and target.std() > 0 and prediction.std() > 0 else float('nan'))
    quantile_levels = (0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99)
    return {
        'count': int(target.size),
        'true_mean_increment': float(target.mean()),
        'pred_mean_increment': float(prediction.mean()),
        'true_mean_rate': float(target.mean() / interval),
        'pred_mean_rate': float(prediction.mean() / interval),
        'mean_ratio_pred_over_true': (float(prediction.mean() / target.mean())
                                      if target.mean() != 0 else float('nan')),
        'mae': float(np.mean(np.abs(residual))),
        'rmse': float(np.sqrt(np.mean(residual ** 2))),
        'bias_pred_minus_true': float(residual.mean()),
        'pearson_r': pearson,
        'r2': (float(1.0 - np.sum(residual ** 2) / centered_ss)
               if centered_ss > 0 else float('nan')),
        'true_positive_fraction': float(np.mean(target > 0)),
        'pred_positive_fraction': float(np.mean(prediction > 0)),
        'true_quantiles': {f'q{int(q * 100):02d}': float(v) for q, v in
                           zip(quantile_levels, np.quantile(target, quantile_levels))},
        'pred_quantiles': {f'q{int(q * 100):02d}': float(v) for q, v in
                           zip(quantile_levels, np.quantile(prediction, quantile_levels))},
    }


def contribution_summary(values: np.ndarray, interval: float, total_mean: float) -> dict:
    values = np.asarray(values, dtype=np.float64).ravel()
    quantile_levels = (0.05, 0.25, 0.50, 0.75, 0.95)
    return {
        'count': int(values.size),
        'mean_increment': float(values.mean()),
        'mean_rate': float(values.mean() / interval),
        'mean_absolute_increment': float(np.mean(np.abs(values))),
        'standard_deviation': float(values.std()),
        'positive_fraction': float(np.mean(values > 0)),
        'fraction_of_predicted_mean': (float(values.mean() / total_mean)
                                       if total_mean != 0 else float('nan')),
        'quantiles': {f'q{int(q * 100):02d}': float(v) for q, v in
                      zip(quantile_levels, np.quantile(values, quantile_levels))},
    }


total_metrics = comparison_metrics(true_total_flat, pred_total_flat, DT_RECORDED)
local_metrics = comparison_metrics(true_local_flat, pred_local_flat, DT_RECORDED)
true_mean_map = true_local.mean(axis=(0, 1))
pred_mean_map = pred_local.mean(axis=(0, 1))
mean_map_metrics = comparison_metrics(true_mean_map, pred_mean_map, DT_RECORDED)
test_objective = alpha_neep_objective(pred_total_flat, training_config.alpha)
predicted_mean = float(pred_total_flat.mean())
component_contribution_summary = {
    key: contribution_summary(component_total_flat[:, index], DT_RECORDED, predicted_mean)
    for index, key in enumerate(CHANNEL_KEYS)
}
if INCLUDE_ANGLE:
    polarization_total_flat = component_total_flat[:, 1:].sum(axis=1)
    polarization_contribution_summary = contribution_summary(
        polarization_total_flat, DT_RECORDED, predicted_mean
    )
else:
    polarization_total_flat = np.zeros_like(pred_total_flat)
    polarization_contribution_summary = None

exact_map_error = np.abs(true_local.sum(axis=(-2, -1)) - true_total)
pred_map_error = np.abs(pred_local.sum(axis=(-2, -1)) - pred_total)
direct_map_error = np.abs(pred_direct - pred_total_flat)
component_additivity_error = np.abs(component_total_flat.sum(axis=1) - pred_total_flat)
scale_audit = {
    'microscopic_dt': float(physics_config.dt),
    'sampling_steps': int(data_config['sampling_steps']),
    'dt_recorded': float(DT_RECORDED),
    'lattice_sites': int(H * W),
    'raw_map_to_local_factor': float(1.0 / (H * W)),
    'model_score_to_total_factor': 1.0,
    'exact_map_sum_max_abs_error': float(exact_map_error.max()),
    'pred_map_sum_max_abs_error': float(pred_map_error.max()),
    'direct_vs_map_total_max_abs_error': float(direct_map_error.max()),
    'component_additivity_max_abs_error': float(component_additivity_error.max()),
    'evaluation_map_scale_quantile': 0.99,
}
accepted_hop_summary = {
    'total': int(accepted_hops.sum()),
    'mean_per_saved_interval': float(accepted_hops.mean()),
    'median_per_saved_interval': float(np.median(accepted_hops)),
    'fraction_intervals_with_hop': float(np.mean(accepted_hops > 0)),
    'min_per_saved_interval': int(accepted_hops.min()),
    'max_per_saved_interval': int(accepted_hops.max()),
}

pooled_abs = np.abs(np.concatenate((true_local_flat.ravel(), pred_local_flat.ravel())))
positive_abs = pooled_abs[pooled_abs > 0]
EP_VMAX = (max(float(np.quantile(positive_abs, 0.99)), 1e-8)
           if positive_abs.size else 1.0)
print(json.dumps({'total_metrics': total_metrics,
                  'component_contributions': component_contribution_summary,
                  'polarization_combined': polarization_contribution_summary,
                  'accepted_hops': accepted_hop_summary,
                  'test_objective': test_objective, 'scale_audit': scale_audit}, indent=2))
print(f'evaluation-map symmetric scale: [-{EP_VMAX:.6g}, +{EP_VMAX:.6g}]')

## 7. Loss history

In [ ]:
history_iterations = np.asarray(training_result.history_iterations)
history_train = np.asarray(training_result.train_losses)
history_validation = np.asarray(training_result.validation_losses)
assert history_iterations.size == history_train.size == history_validation.size > 0

fig, ax = plt.subplots(figsize=(7.2, 4.2), constrained_layout=True)
ax.plot(history_iterations, history_train, 'o-', ms=3, label='train minibatch')
ax.plot(history_iterations, history_validation, 'o-', ms=3, label='independent validation')
ax.axhline(0.0, color='black', lw=0.8, alpha=0.5, label='zero-score baseline')
ax.axvline(training_result.best_iteration, color='tab:green', ls='--', lw=1, label='best iteration')
ax.set(xlabel='iteration', ylabel='alpha-NEEP objective', title='ShellForce KNEEP training')
ax.grid(alpha=0.25)
ax.legend()
fig.savefig(RESULT_DIR / 'loss.png', dpi=170)
plt.show()

## 8. Held-out true-versus-predicted EP performance

The time series uses only test replica 0 and a moving average for readability; the scatter and printed metrics use every held-out pair. There is intentionally no cumulative-EP plot. Microscopic medium EP and joint-state two-frame KNEEP EP need not agree pair by pair: the latter can contain system-boundary/angular information, while coarse saved frames can hide microscopic path information.

In [ ]:
def moving_average(values: np.ndarray, window: int) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    window = max(1, min(int(window), values.size))
    return np.convolve(values, np.ones(window) / window, mode='same')


smooth_window = max(1, min(25, P // 8))
scatter_ids = animation_frames(M * P, maximum=5000)
fig, axes = plt.subplots(1, 2, figsize=(13.2, 4.8), constrained_layout=True)
axes[0].plot(time_mid, moving_average(true_total[0], smooth_window),
             label=f'true microscopic (MA {smooth_window})')
axes[0].plot(time_mid, moving_average(pred_total[0], smooth_window),
             label=f'KNEEP joint-state (MA {smooth_window})')
axes[0].set(xlabel='time', ylabel='EP per saved transition',
            title='Held-out test trajectory 0')
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].scatter(true_total_flat[scatter_ids], pred_total_flat[scatter_ids], s=13, alpha=0.4)
lo = float(min(true_total_flat.min(), pred_total_flat.min()))
hi = float(max(true_total_flat.max(), pred_total_flat.max()))
if hi <= lo:
    hi = lo + 1.0
axes[1].plot([lo, hi], [lo, hi], 'k--', lw=1, label='identity')
axes[1].set(xlabel='true microscopic medium-EP increment',
            ylabel='predicted joint-state EP increment',
            title=f"All test pairs: r={total_metrics['pearson_r']:.3f}, R2={total_metrics['r2']:.3f}")
axes[1].grid(alpha=0.25)
axes[1].legend()
fig.savefig(RESULT_DIR / 'true_vs_pred_ep.png', dpi=170)
plt.show()
print(json.dumps(total_metrics, indent=2))

## 9. Predicted local EP animation

This uses the same frame indices as the earlier true-map animation and the pooled true/predicted evaluation scale computed above. Each predicted map sums the contracted $\rho$, $\rho\cos\theta$, and $\rho\sin\theta$ component maps when angle is enabled; its map sum equals the predicted total by assertion above. Learned pixels and their component split should not be read as destination-gauge or unique thermodynamic labels.

In [ ]:
pred_ep_animation, pred_ep_html = make_ep_animation(
    pred_local[0], 'predicted joint-state EP (learned gauge)', EP_VMAX,
    'predicted EP increment per learned site',
)
display(pred_ep_html)

## 10. Time-mean local-map diagnostic

The first two panels share the animation color scale. The pixelwise residual and metrics are gauge-dependent localization diagnostics, not a thermodynamic equality test.

In [ ]:
mean_residual = pred_mean_map - true_mean_map
residual_abs = np.abs(mean_residual)
residual_vmax = max(float(np.quantile(residual_abs[residual_abs > 0], 0.99)), 1e-8) \
    if np.any(residual_abs > 0) else 1.0
fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.2), constrained_layout=True)
im0 = axes[0].imshow(true_mean_map.T, origin='lower', cmap='RdBu_r', vmin=-EP_VMAX, vmax=EP_VMAX)
im1 = axes[1].imshow(pred_mean_map.T, origin='lower', cmap='RdBu_r', vmin=-EP_VMAX, vmax=EP_VMAX)
im2 = axes[2].imshow(mean_residual.T, origin='lower', cmap='RdBu_r',
                     vmin=-residual_vmax, vmax=residual_vmax)
axes[0].set_title('time-mean true (destination gauge)')
axes[1].set_title('time-mean predicted (learned gauge)')
axes[2].set_title(f"pred - true (pixel r={mean_map_metrics['pearson_r']:.3f})")
fig.colorbar(im0, ax=axes[:2], shrink=0.8, label='mean EP increment')
fig.colorbar(im2, ax=axes[2], shrink=0.8, label='mean residual')
for ax in axes:
    ax.set(xticks=[], yticks=[])
fig.savefig(RESULT_DIR / 'time_mean_local_ep.png', dpi=170)
plt.show()
print('all-pair local pixel metrics:')
print(json.dumps(local_metrics, indent=2))
print('time-mean map pixel metrics:')
print(json.dumps(mean_map_metrics, indent=2))

## 11. Shell spectrum and component-resolved predicted EP

Each spectrum point is the held-out mean of one ShellForce branch total. The linear panel preserves sign; the log panel shows absolute magnitude and retains sign through color. A second figure exposes the contracted $\rho$, $\rho\cos\theta$, and $\rho\sin\theta$ contributions over time and in held-out means. Both decompositions are descriptive and coordinate dependent. No cumulative spectrum is computed.

In [ ]:
shell_indices = np.arange(K)
mean_shell_ep = branch_total_flat.mean(axis=0)
shell_magnitude = np.abs(mean_shell_ep)
nonzero_shell = shell_magnitude[shell_magnitude > 0]
log_floor = float(nonzero_shell.min() * 0.5) if nonzero_shell.size else 1e-12
log_heights = np.maximum(shell_magnitude, log_floor)
shell_colors = np.where(mean_shell_ep >= 0, 'tab:blue', 'tab:red')
np.testing.assert_allclose(branch_total_flat.sum(axis=1), pred_total_flat, rtol=2e-5, atol=2e-5)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.5), constrained_layout=True)
axes[0].bar(shell_indices, mean_shell_ep, color=shell_colors)
axes[0].axhline(0.0, color='black', lw=0.8)
axes[0].set(xlabel='exclusive Chebyshev shell distance', ylabel='mean signed EP increment',
            title='ShellForce EP spectrum (signed)')
axes[0].set_xticks(shell_indices)
axes[0].grid(axis='y', alpha=0.25)

axes[1].bar(shell_indices, log_heights, color=shell_colors)
axes[1].set_yscale('log')
axes[1].set(xlabel='exclusive Chebyshev shell distance', ylabel='|mean EP increment|',
            title='ShellForce EP spectrum (magnitude, log scale)')
axes[1].set_xticks(shell_indices)
axes[1].grid(axis='y', which='both', alpha=0.25)
for index, value in enumerate(mean_shell_ep):
    axes[1].text(index, log_heights[index], '+' if value >= 0 else '-', ha='center', va='bottom')
fig.savefig(RESULT_DIR / 'shell_ep_spectrum.png', dpi=170)
plt.show()
print(dict(zip(shell_indices.tolist(), mean_shell_ep.tolist())))

mean_component_ep = component_total_flat.mean(axis=0)
mean_abs_component_ep = np.mean(np.abs(component_total_flat), axis=0)
component_positions = np.arange(Q)
component_colors = ['tab:gray', 'tab:orange', 'tab:green'][:Q]
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.5), constrained_layout=True)
for index, name in enumerate(CHANNEL_NAMES):
    axes[0].plot(time_mid, moving_average(component_total[0, :, index], smooth_window),
                 color=component_colors[index], label=f'{name} contribution')
axes[0].plot(time_mid, moving_average(pred_total[0], smooth_window), 'k--', lw=1.2,
             label='sum')
axes[0].set(xlabel='time', ylabel='predicted EP per saved transition',
            title=f'Component contributions, test trajectory 0 (MA {smooth_window})')
axes[0].grid(alpha=0.25)
axes[0].legend()

width = 0.36
axes[1].bar(component_positions - width / 2, mean_component_ep, width,
            color=component_colors, label='signed mean')
axes[1].bar(component_positions + width / 2, mean_abs_component_ep, width,
            color=component_colors, alpha=0.45, hatch='//', label='mean absolute')
axes[1].axhline(0.0, color='black', lw=0.8)
axes[1].set(xlabel='contracted observation component', ylabel='EP increment',
            title='Held-out component contribution summary')
axes[1].set_xticks(component_positions, CHANNEL_NAMES, rotation=12)
axes[1].grid(axis='y', alpha=0.25)
axes[1].legend()
fig.savefig(RESULT_DIR / 'component_ep_contributions.png', dpi=170)
plt.show()
print(json.dumps({'by_component': component_contribution_summary,
                  'polarization_combined': polarization_contribution_summary}, indent=2))

## 12. Summary and reproducible outputs

This cell saves `summary.json`, the PNG plots, and compressed held-out arrays under `results/tc_labp_demo/<timestamp>/`. Animations are intentionally inline JSHTML only. Non-finite diagnostic values (for example an undefined Pearson correlation in a degenerate smoke run) are written as JSON `null`.

In [ ]:
summary = {
    'schema_version': 5,
    'generated_at': RUN_STAMP,
    'device': str(DEVICE),
    'trajectory_data': {
        'loaded_from': TRAJECTORY_LOADED_FROM,
        'saved_to': TRAJECTORY_SAVED_TO,
    },
    'angle': {
        'included': bool(INCLUDE_ANGLE),
        'channels': CHANNEL_NAMES,
        'ep_component_indices': list(EP_COMPONENT_INDICES),
        'semantics': ('all encoded state increments are contracted' if INCLUDE_ANGLE
                      else 'occupancy is the only encoded and contracted state'),
    },
    'model': {'type': 'ShellForceKNEEP2D', **model_config},
    'physics': {
        **asdict(physics_config),
        'n_particles': int(physics_config.n_particles),
        'Pe': float(physics_config.Pe),
        'max_hop_probability': float(physics_config.max_hop_probability),
    },
    'data': {**data_config, 'seeds': SEEDS, 'independent_train_validation_test': True},
    'training': {
        **asdict(training_config),
        'best_iteration': int(training_result.best_iteration),
        'best_validation_loss': float(training_result.best_validation_loss),
        'test_objective': float(test_objective),
        'zero_score_objective': 0.0,
    },
    'total_increment_metrics': total_metrics,
    'predicted_component_contributions_coordinate_dependent': {
        'component_order': CHANNEL_KEYS,
        'by_component': component_contribution_summary,
        'polarization_combined': polarization_contribution_summary,
        'interpretation': ('learned score split, not a unique system-vs-medium EP split'),
    },
    'all_pair_local_pixel_metrics_gauge_dependent': local_metrics,
    'time_mean_local_pixel_metrics_gauge_dependent': mean_map_metrics,
    'accepted_hops': accepted_hop_summary,
    'scale_and_map_sum_audit': {
        **scale_audit,
        'true_animation_vmax': float(TRUE_EP_VMAX),
        'evaluation_map_vmax': float(EP_VMAX),
    },
    'shell_spectrum': {
        'distance': shell_indices.tolist(),
        'mean_signed_increment': mean_shell_ep.tolist(),
        'mean_magnitude_for_log_plot': shell_magnitude.tolist(),
        'cumulative_computed': False,
    },
    'semantics': {
        'true_ep': 'accepted-hop microscopic reservoir/medium EP over each saved interval',
        'true_system_ep_included': False,
        'prediction_uses_joint_observed_state': bool(INCLUDE_ANGLE),
        'prediction_can_represent_angular_system_boundary_effects': bool(INCLUDE_ANGLE),
        'orientation_time_reversal_parity': 'even',
        'exact_system_ep_separately_identified': False,
        'true_local_gauge': 'destination site',
        'pred_local_gauge': 'learned convolution-center gauge',
        'pairwise_label_claim': False,
    },
}


def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


summary_json = json_safe(summary)
(RESULT_DIR / 'summary.json').write_text(
    json.dumps(summary_json, indent=2, allow_nan=False), encoding='utf-8'
)
np.savez_compressed(
    RESULT_DIR / 'held_out_ep.npz',
    times=times,
    true_total=true_total,
    pred_total=pred_total,
    pred_component_total=component_total,
    pred_polarization_total=polarization_total_flat.reshape(M, P),
    true_local=true_local,
    pred_local=pred_local,
    pred_component_local=component_local,
    accepted_hops=accepted_hops,
    raw_branch_totals=branch_total_flat.reshape(M, P, K),
    raw_component_branch_totals=component_branch_total_flat.reshape(M, P, K, Q),
    component_indices=np.asarray(EP_COMPONENT_INDICES, dtype=np.int64),
    component_names=np.asarray(CHANNEL_KEYS),
    shell_distance=shell_indices,
    mean_shell_ep=mean_shell_ep,
    history_iteration=history_iterations,
    history_train_loss=history_train,
    history_validation_loss=history_validation,
)

print(json.dumps(summary_json, indent=2, allow_nan=False))
print('saved files:')
for path in sorted(RESULT_DIR.iterdir()):
    print(' -', path.name)